[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/REPO/blob/BRANCH/PATH/TO/NOTEBOOK.ipynb) **Remember to complete the github path to make this link work**


# Getting started: intro to FiftyOne datasets

## Content overview

In this tutorial we cover the following concepts:

- [Dataset basics and samples](https://beta-docs.voxel51.com/getting_started/basic/datasets_samples_fields/)
- [Detection fields and labels](https://beta-docs.voxel51.com/api/fiftyone.core.labels.Detection.html)
- [Dataset views and filtering](https://beta-docs.voxel51.com/how_do_i/cheat_sheets/filtering_cheat_sheet/)


## Who is this for

This tutorial is designed for:

* Computer vision practitioners who are new to the FiftyOne app
* Anyone looking to integrate annotations into their visual datasets for analysis

## Assumed knowledge

- Basic knowledge of image processing
- Intermediate Python programming
- Experience working with Jupyter notebooks

## Time to complete
Estimated time: 30-45 minutes

## Required packages for local installation

If not running from a Google Colab environment, w recommend using a virtual environment with [FiftyOne installed](https://beta-docs.voxel51.com/getting_started/basic/install/).



Run this cell to install FiftyOne on Google Colab

In [ ]:
!pip install fiftyone==1.4.0 -q > /dev/null

Import the FiftyOne library and check the version that has been installed.

In [ ]:
import fiftyone as fo
fo.__version__

## Inspecting the quickstart `Dataset`

In the FiftyOne dataset zoo, we have a collection of publicly available datasets that can be easily loaded and used for computer vision tasks. The `quickstart` dataset is a small dataset of 200 images with ground truth annotations and object detections that is commonly used for demonstrating the features of FiftyOne.

Import the FiftyOne zoo for loading datasets.

Load the "quickstart" dataset from the FiftyOne zoo. When we specify `persistent = True`, we make sure that changes to the dataset are saved across multiple Python sessions.

In [ ]:
import fiftyone.zoo as foz

dataset = foz.load_zoo_dataset("quickstart", persistent=True)

We can use the `stats()` method on a dataset to obtain info about its number of samples and size on disk.

In [ ]:
dataset.stats()

The `info` field in the dataset can be used to add extra metadata. Here we are using it to specify where we got the dataset from and its license (Creative Commons 4.0).

In [ ]:
dataset.info["dataset_source"] = "https://docs.voxel51.com/dataset_zoo/datasets.html#dataset-zoo-quickstart"
dataset.info["dataset_license"] = "CC-BY-4.0"
dataset.info

`.count()` tells us how many samples have been added to the dataset.

In [ ]:
dataset.count()

In FiftyOne, a sample is an image and all its associated tags, metadata, and annotations.

In [ ]:
sample=dataset.first()
sample

## FiftyOne sample fields

In FiftyOne, a field is an attribute associated with each sample (e.g., image or video) in a dataset. Fields store labels, metadata, predictions, or custom data. Fields provide a way to organize and access information about your data within the FiftyOne framework. You can use fields to filter, sort, and analyze your dataset, and they play a role in tasks like model evaluation and data visualization. Some examples of built-in fields are `filepath`, `ground_truth`, and `predictions`, but you can also define your own custom fields to store any data you need.



Here we can inspect the fields of the sample.

In [ ]:
sample.field_names

The ID of the sample is a hash that is unique for each image.

In [ ]:
sample.id

We have a datetime object specifying when the dataset was created by us on disk.

In [ ]:
sample.created_at

`sample.ground_truth` specifies the labels and positions of our object detections.

In [ ]:
sample.ground_truth

`sample.predictions` will give us the list of all bounding boxes that have been computed on the dataset already. Note that each detection has a unique hash ID, a label, a confidence level, and an associated bounding box.

In [ ]:
sample.predictions

Notice that the `sample.filepath` points to the path to the image on the hard drive. `filepath` is the only required sample field when creating our own datasets.

In [ ]:
sample.filepath

As the file is local, we can open it with PIL, NumPy or PyTorch.

In [ ]:
from PIL import Image
original_sample = Image.open(sample.filepath)
original_sample

We can add fields to our sample, in order to extend it.

In [ ]:
sample['inspected_in_notebook'] = True
sample.field_names

We can also add the field at the dataset level.

In [ ]:
dataset.add_sample_field("inspected_in_notebook", fo.BooleanField)

In FiftyOne, files are loaded from disk. The images from the `quickstart` dataset are now on the hard drive of the computer running this notebook.

In [ ]:
from pathlib import Path
!ls {Path(sample.filepath).parent}

When evaluating the dataset object, we get a quick look at its attributes.

In [ ]:
dataset

Our dataset consists of images and we have 200 samples of them.

In [ ]:
dataset.media_type, len(dataset)

Launch the FiftyOne App and load the full dataset into it.

In [ ]:
session = fo.launch_app(dataset)

# Clone vs. Views in FiftyOne Datasets

In FiftyOne, both clones and views provide ways to work with datasets, but they serve different purposes:

## Clones

A clone is a complete copy of a dataset. When we clone a dataset in FiftyOne:

- Changes made to the clone do not affect the original dataset
- Clones are independent datasets with their own names in FiftyOne's MongoDB database
- All samples, fields, and metadata are fully copied into a clone

Here's how we would create a clone:




Create a clone.

In [ ]:
cloned_dataset = dataset.clone("my-quickstart-clone")

The clone will now appear in the list of available datasets.

In [ ]:
fo.list_datasets()

In [ ]:
cloned_dataset.first().filepath

Create a new sample.

In [ ]:
grayscale_sample =  Image.open(cloned_dataset.first().filepath).convert("L")

In [ ]:
cloned_dataset

## Views

A view is a filtered subset of the FiftyOne dataset. When when we create a view:

- Changes to samples in the view will affect the original dataset
- It's not a separate dataset but a lens into the original dataset
- No data is duplicated; it's just a different way to access the source dataset
- Views are memory-efficient since they don't copy data
- Views can apply filters, sorting, and other operations to show only specific data

Views are useful to retrieve a subset of data, it's good to check out the [views cheat sheet](https://docs.voxel51.com/cheat_sheets/views_cheat_sheet.html).




## Slicing to create views

Slicing is a way to create dataset views. Here we select three samples from the dataset, starting at index 7.

In [ ]:
dataset[7:10]

# `ViewField` in FiftyOne

A `ViewField` in FiftyOne is a dynamic field that computes its values on-the-fly rather than storing them directly in the dataset. Unlike regular fields that permanently store data, ViewFields are computed when accessed and don't persist any values to disk.

## Key characteristics of ViewFields

1. **Dynamic Computation**: Values are generated at access time through a user-defined function
2. **Non-persistent**: The values aren't stored in the database
3. **Memory Efficient**: Since values aren't stored, they don't increase dataset storage requirements
4. **Function-based**: Each `ViewField` is backed by a Python function that determines its values

## How ViewFields Work

When you define a `ViewField`, you provide a function that specifies how to compute the field's value for each sample. The function typically takes a sample as input and returns the desired value based on other fields in that sample.

Here's a basic example of creating a `ViewField` to filter out the images with cats in our quickstart data.


### Using `match_labels()` to filter samples

The `match_labels()` method is used to create a `DatasetView` containing only the **samples** that have *at least one* label matching a specified filter condition within designated label fields. In the cell below you see it being used to select images labeled as "cat".

In [ ]:
from fiftyone import ViewField as F

# Use match_labels to filter samples that contain at least one "cat" label
cats_view = (
    dataset
    .match_labels(fields="ground_truth", filter=F("label") == "cat") # Keep only samples with "cat" detections
)

# Optional: You can verify the number of samples in the view
print(f"Number of samples containing cats: {len(cats_view)}")

We can always re-launch the app with a view, allowing us to see the filtered dataset.

In [ ]:
fo.launch_app(cats_view)

## Differences between a dataset clone and a view

Think of a dataset as a library of books:
- **Clone**: Making a complete duplicate of the library in a new building. Changes to one library don't affect the other.
- **View**: Creating a reading list that references specific books in the original library. If you write notes in a book from the reading list, those notes appear in the original library's book too.

## When to Use Each

- Use **clones** when you need a complete backup or want to make extensive changes without affecting the original data
- Use **views** when you need to temporarily filter, sort, or manipulate a dataset without duplicating data or when you want changes to propagate to the original dataset

Does this explanation help clarify the differences between clones and views in FiftyOne?

## Stats on a `DatasetView`

We can use `stats()` on a dataset view to retrieve info on a subset of the data.

In [ ]:
cats_view.stats()

## Summary

This tutorial covered:

- Loading datasets from the FiftyOne Dataset Zoo.
- Inspecting dataset samples and fields.
- Cloning datasets for independent copies.
- Creating dataset views for filtering and sorting.
- Using [`ViewField`]() for dynamic computations.
- Using [match_labels()](https://docs.voxel51.com/api/fiftyone.core.collections.html#fiftyone.core.collections.SampleCollection.match_labels) to filter samples.
- Launching the FiftyOne app from different dataset subsets and views.

